# Module 06: Distributed Compute with Ray

## What You'll Learn

- Why distributed compute matters for feature stores
- Feast's Ray compute engine architecture
- Three execution modes: LOCAL, REMOTE, KUBERAY
- Distributed materialization with Ray
- Ray + Feast for embedding generation (RAG pipeline preview)
- KubeRay integration on RHOAI

---

> **🗺️ DATA STRATEGY**: This module covers **Pillar 2: Compute Engine Strategy**. The data strategy positions Ray Data as the Python-native distributed engine for AI/ML workloads. Feast+Ray productization is Phase 2 of the compute strategy. This is the dominant customer demand signal (P2 in the client-pillar mapping).

## Why Distributed Compute for Feast?

The default Feast materialization engine processes features **sequentially**. For production workloads:
- Large entity sets (millions of customers)
- Complex transformations (aggregations, joins)
- Embedding generation (GPU-accelerated)

...you need distributed compute.

Feast supports pluggable compute engines:

| Engine | Best For | RHOAI Status |
|--------|----------|-------------|
| **Local** | Development, small datasets | Default |
| **Ray** | Python-native ML workloads, embeddings | Available upstream (feast-dev/feast#5526) |
| **Spark** | SQL-heavy ETL, large structured datasets | Community plugin |

> **⚠️ GAP**: Feast+Ray integration exists upstream but is **not productized downstream** in RHOAI. Users must configure manually. The operator does not expose compute engine selection in the CRD. This is a P0 gap.
>
> **🔮 UPSTREAM**: [feast-dev/feast#5526](https://github.com/feast-dev/feast/pull/5526) merged Ray as a distributed compute engine. Three deployment modes, GPU support for embeddings, and parallel materialization.

## Ray Compute Engine Architecture

Feast's Ray engine uses a DAG-based architecture:

```
Pipeline DAG:
  EntityDF
    → RayReadNode (parallel reads from offline store)
    → RayJoinNode (distributed point-in-time joins)
    → RayFilterNode (parallel filtering)
    → RayAggregationNode (distributed aggregations)
    → RayTransformationNode (parallel transformations)
    → Output (materialized features)
```

Each node in the DAG maps to a Ray task that runs on the cluster.

## Configuration

### Mode 1: LOCAL (Ray runs on the same machine)

```yaml
# feature_store.yaml
batch_engine:
  type: ray
  # No ray_address = LOCAL mode (starts local Ray runtime)
```

### Mode 2: REMOTE (Connect to existing Ray cluster)

```yaml
batch_engine:
  type: ray
  ray_address: ray://raycluster-head.my-namespace.svc:10001
```

### Mode 3: KUBERAY (Connect via CodeFlare SDK to KubeRay)

```yaml
batch_engine:
  type: ray
  ray_address: kuberay://raycluster-head.my-namespace.svc:10001
  # Or via CodeFlare SDK:
  # Uses the codeflare-sdk to discover and connect to KubeRay clusters
```

> **📍 RHOAI STATUS**: KubeRay operator is fully operational in RHOAI with Kueue integration. The Ray cluster infrastructure is ready — the gap is Feast operator awareness of it.

In [ ]:
# Demonstrate Ray compute engine in LOCAL mode
# (This works without a Ray cluster - uses local Ray runtime)

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Generate larger dataset to see Ray benefits
np.random.seed(42)
os.makedirs("data", exist_ok=True)

n_customers = 1000
n_weeks = 52

records = []
for customer_id in range(1, n_customers + 1):
    for week in range(n_weeks):
        ts = datetime(2024, 1, 1) + timedelta(weeks=week)
        records.append({
            "customer_id": customer_id,
            "event_timestamp": ts,
            "credit_score": np.random.randint(300, 850),
            "transaction_count": np.random.randint(0, 500),
            "avg_amount": round(np.random.uniform(10, 5000), 2),
        })

df = pd.DataFrame(records)
df.to_parquet("data/large_features.parquet")
print(f"Generated {len(df):,} records ({n_customers} customers x {n_weeks} weeks)")

In [ ]:
# Configure feature store with Ray compute engine
import yaml

ray_config = {
    "project": "ray_demo",
    "provider": "local",
    "registry": {
        "registry_type": "sql",
        "path": "sqlite:///data/registry.db",
    },
    "offline_store": {"type": "duckdb"},
    "online_store": {"type": "sqlite", "path": "data/online.db"},
    "batch_engine": {
        "type": "ray",
        # No ray_address = LOCAL mode
    },
    "entity_key_serialization_version": 3,
}

os.makedirs("feature_repo", exist_ok=True)
with open("feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(ray_config, f, default_flow_style=False)

print("Configured Feast with Ray compute engine (LOCAL mode)")
print("\nFor RHOAI cluster mode, change to:")
print("  batch_engine:")
print("    type: ray")
print("    ray_address: ray://raycluster-head.my-ns.svc:10001")

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Float32, Int64

customer = Entity(name="customer", join_keys=["customer_id"])
source = FileSource(
    name="large_features_source",
    path=os.path.abspath("data/large_features.parquet"),
    timestamp_field="event_timestamp",
)
fv = FeatureView(
    name="customer_features",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="transaction_count", dtype=Int64),
        Field(name="avg_amount", dtype=Float32),
    ],
    source=source,
)

store = FeatureStore(repo_path="feature_repo")
store.apply([customer, source, fv])
print("✅ Feature store applied with Ray compute engine")

In [ ]:
# Materialize using Ray (distributed)
import time

start = time.time()
store.materialize(
    start_date=datetime(2024, 1, 1),
    end_date=datetime.now(),
)
elapsed = time.time() - start

print(f"✅ Materialization complete in {elapsed:.1f}s (Ray LOCAL mode)")
print(f"\nWith a KubeRay cluster on RHOAI:")
print(f"  - Workers parallelize across nodes")
print(f"  - GPU workers available for embedding generation")
print(f"  - Kueue manages resource allocation")

## Ray for Embedding Generation (RAG Preview)

The Feast+Ray integration is particularly powerful for RAG pipelines:
- Distributed document processing (parallel chunking)
- Parallel embedding generation across GPU workers
- Materialization into vector-enabled online stores

```python
# Example: Feast + Ray + Docling for RAG
# (Full coverage in Module 08)

# feast init -t ray_rag  # Creates a complete Ray RAG template

# The pipeline:
# 1. Documents read in parallel (Ray workers)
# 2. Docling processes each document (text extraction, chunking)
# 3. Embedding model runs on GPU workers (sentence-transformers)
# 4. Results materialized to Milvus/pgvector online store
```

> **🔮 UPSTREAM**: `feast init -t ray_rag` provides a complete template for distributed RAG pipelines. The DocEmbedder class (v0.62.0) streamlines embedding workflows.

## Connecting to KubeRay on RHOAI

```python
# Using CodeFlare SDK to connect to a RayCluster on RHOAI
from codeflare_sdk import RayClusterClient

# Discover available Ray clusters in the namespace
client = RayClusterClient()
clusters = client.list_clusters()
print(f"Available Ray clusters: {clusters}")

# The Feast batch_engine config then uses the Ray address:
# batch_engine:
#   type: ray
#   ray_address: ray://raycluster-head.my-ns.svc:10001
```

> **📍 RHOAI STATUS**: KubeRay is fully operational in RHOAI. CodeFlare SDK provides cluster discovery. The missing piece is operator integration — the Feast operator should be able to submit RayJob CRs for materialization instead of using `oc exec`.
>
> **⚠️ GAP**: No compute engine selection in FeatureStore CRD. Only CronJob-based materialization is operator-managed. Ray/Spark must be configured manually.

## Key Takeaways

1. **Default materialization doesn't scale** — Ray provides distributed compute for production
2. **Three modes**: LOCAL (dev), REMOTE (existing cluster), KUBERAY (on RHOAI via CodeFlare)
3. **Ray+Feast is the data strategy's compute answer** for AI/ML-native workloads
4. **Embedding generation** benefits most from Ray's GPU-parallel workers
5. **Gap**: Not yet operator-managed — manual config required on RHOAI

## What's Next

- **Module 07**: OpenLineage — lineage tracking (materialization emits lineage events)
- **Module 08**: RAG & Vector Search — full Ray+Docling+Embeddings pipeline